# 02 · R2: calibration fine-tune for Phase 4 (+ R3 on the idle GPU)

**GPU0:** fine-tune `lerobot/smolvla_base` (paper architecture, VLM frozen) on **LIBERO-Goal task 0 only**,
saving a checkpoint every `SAVE_FREQ` steps; then each checkpoint is evaluated on task 0 (10 episodes).
This tells Phase 4 how many steps one stage needs to reach ≥ 70 % success (gate G1).

**GPU1 (while GPU0 trains):**
1. **R1b** — `lerobot/smolvla_libero`, the LeRobot checkpoint with the *paper's* architecture (16 VLM layers, 0.75× expert,
   fine-tuned from `smolvla_base`), on all 10 Goal tasks × 10 episodes with `n_action_steps = 10` (best value in paper Table 13).
2. **R3** — `HuggingFaceVLA/smolvla_libero` with `n_action_steps` = 10 and 50 on Goal tasks 0 and 9
   (R1 already has `n_action_steps` = 1), evidence for the open-loop-chunk limitation.

Run with **Save Version → Save & Run All (Commit)**, GPU T4 x2, Internet On.
Outputs in `/kaggle/working/r2/`: `summary.md`, `progress.log`, `train.log`, `eval/*/eval_info.json`, `train/checkpoints/*/pretrained_model`.

In [ ]:
# ---- parameters ----
SUITE, TASK = "libero_goal", 0
STEPS, SAVE_FREQ, BATCH = 1500, 500, 32   # T4 ~9.4 s/step at batch 32 (no native bf16) -> ~4 h
EVAL_EPISODES = 10
EVAL_NAS = 10                  # n_action_steps for the checkpoint sweep (= Phase 4 eval setting)
FINAL_NAS1 = True              # also evaluate the last checkpoint with n_action_steps = 1
R1B = False                    # done in run 2 (88/100)
R1B_CKPT, R1B_NAS = "lerobot/smolvla_libero", 10
R3 = False                     # done in run 2
R3_TASKS, R3_NAS = [0, 9], [10, 50]
open("/tmp/params.sh", "w").write(
    f'SUITE={SUITE}\nTASK={TASK}\nSTEPS={STEPS}\nSAVE_FREQ={SAVE_FREQ}\nBATCH={BATCH}\n'
    f'EVAL_EPISODES={EVAL_EPISODES}\nEVAL_NAS={EVAL_NAS}\nFINAL_NAS1={int(FINAL_NAS1)}\nR3={int(R3)}\n'
    f'R1B={int(R1B)}\nR1B_CKPT={R1B_CKPT}\nR1B_NAS={R1B_NAS}\n'
    f'R3_TASKS="{" ".join(map(str, R3_TASKS))}"\nR3_NAS="{" ".join(map(str, R3_NAS))}"\n')
print(open("/tmp/params.sh").read())

In [ ]:
%%bash
echo "=== ENV ===" >> /kaggle/working/r2_report.txt
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv | tee -a /kaggle/working/r2_report.txt
echo "CPUs: $(nproc)" | tee -a /kaggle/working/r2_report.txt
free -g | head -2 | tee -a /kaggle/working/r2_report.txt

In [ ]:
%%bash
# System libraries for headless MuJoCo rendering and video decoding
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libegl1 libgl1 libglvnd0 libosmesa6 libosmesa6-dev ffmpeg cmake build-essential > /dev/null 2>&1 \
  && echo "apt: ok" | tee -a /kaggle/working/r2_report.txt || { echo "apt: FAILED" | tee -a /kaggle/working/r2_report.txt; exit 1; }
# LeRobot needs Python >= 3.12; build an isolated venv with uv
pip install -q -U uv
uv python install 3.12 > /dev/null 2>&1
rm -rf /tmp/venv && uv venv /tmp/venv --python 3.12 > /dev/null 2>&1
/tmp/venv/bin/python --version > /dev/null 2>&1 \
  && echo "venv: ok ($(/tmp/venv/bin/python --version))" | tee -a /kaggle/working/r2_report.txt || { echo "venv: FAILED" | tee -a /kaggle/working/r2_report.txt; exit 1; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
LOG=/kaggle/working/install.log
cd /tmp && rm -rf lerobot && git clone -q https://github.com/huggingface/lerobot.git && cd lerobot && git checkout -q ba83e4cf43737e2665be0aafcb5f513b83c7d0f3
echo "lerobot commit: $(git rev-parse HEAD)" | tee -a /kaggle/working/r2_report.txt
unset UV_PYTHON UV_PROJECT_ENVIRONMENT PIP_TARGET
# Target the venv explicitly: Kaggle's environment otherwise sends uv to the system Python
uv pip install --python /tmp/venv/bin/python -e ".[smolvla,libero]" > $LOG 2>&1
RC=$?
echo "install exit: $RC ($(head -1 $LOG))" | tee -a /kaggle/working/r2_report.txt
if [ $RC -ne 0 ]; then tail -40 $LOG | tee -a /kaggle/working/r2_report.txt; exit 1; fi
python - <<'EOF' 2>&1 | tee -a /kaggle/working/r2_report.txt
import importlib, torch
for m in ["torch", "lerobot", "transformers", "mujoco", "libero"]:
    mod = importlib.import_module(m)
    print(f"import {m}: ok {getattr(mod, '__version__', '')}")
print("cuda:", torch.version.cuda, "|", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
EOF
[ ${PIPESTATUS[0]} -eq 0 ] || exit 1
# LIBERO asks for a dataset path on first import; answer "N" to keep defaults
(yes N | python -c "from libero.libero import benchmark; print('libero:', len(benchmark.get_benchmark_dict()), 'suites')") 2>&1 | tail -1 | tee -a /kaggle/working/r2_report.txt
# Kaggle ships its own /usr/local/bin/lerobot-* that shadow the venv's; every call below uses `/tmp/venv/bin/python -m ...`
echo "lerobot-eval on PATH: $(which -a lerobot-eval | tr '\n' ' ')" | tee -a /kaggle/working/r2_report.txt
/tmp/venv/bin/python -c "import lerobot, libero, mujoco" && touch /tmp/setup_ok && echo "setup: ok" | tee -a /kaggle/working/r2_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
# Episodes of each LIBERO-Goal task in lerobot/libero at a pinned revision. Episode metadata carries no task,
# so the map was built offline from the per-frame `task_index` (kaggle/libero_goal_episodes.json).
mkdir -p /kaggle/working/r2
cat > /tmp/libero_goal_episodes.json <<'EOF'
{"revision": "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4", "tasks": {"0": {"instruction": "open the middle drawer of the cabinet", "task_index": 19, "episodes": [399, 405, 410, 421, 437, 443, 463, 469, 473, 490, 496, 497, 516, 519, 525, 541, 547, 552, 561, 566, 577, 585, 599, 611, 631, 633, 659, 664, 672, 678, 687, 693, 703, 722, 725, 734, 754, 765, 772, 779, 780, 792, 802]}, "1": {"instruction": "put the bowl on the stove", "task_index": 17, "episodes": [392, 394, 395, 401, 415, 424, 445, 455, 495, 500, 524, 530, 544, 545, 550, 557, 558, 565, 580, 598, 604, 612, 616, 618, 629, 637, 641, 643, 649, 673, 676, 688, 691, 692, 704, 707, 721, 740, 741, 747, 751, 763, 764, 786, 789, 796, 804, 805]}, "2": {"instruction": "put the wine bottle on top of the cabinet", "task_index": 14, "episodes": [385, 389, 396, 397, 404, 417, 419, 423, 427, 430, 432, 436, 454, 468, 470, 475, 476, 479, 480, 482, 512, 515, 528, 533, 543, 546, 554, 555, 556, 574, 588, 591, 592, 596, 619, 635, 645, 648, 656, 668, 682, 686, 697, 735, 766, 788, 795]}, "3": {"instruction": "open the top drawer and put the bowl inside", "task_index": 12, "episodes": [382, 383, 387, 409, 414, 416, 449, 462, 465, 489, 494, 498, 523, 526, 534, 538, 548, 559, 560, 576, 594, 614, 624, 627, 684, 694, 701, 711, 737, 738, 742, 758, 762, 782, 784, 790]}, "4": {"instruction": "put the bowl on top of the cabinet", "task_index": 18, "episodes": [398, 402, 438, 444, 446, 452, 456, 458, 474, 493, 501, 505, 514, 521, 570, 573, 590, 597, 601, 608, 630, 636, 638, 660, 661, 662, 665, 669, 674, 681, 689, 696, 699, 702, 706, 712, 724, 731, 732, 743, 745, 767, 775, 791, 793, 794]}, "5": {"instruction": "push the plate to the front of the stove", "task_index": 15, "episodes": [386, 400, 411, 420, 425, 442, 461, 471, 472, 478, 506, 535, 553, 584, 589, 593, 603, 606, 647, 663, 680, 685, 710, 713, 715, 733, 744, 752, 755, 761, 769, 778, 781]}, "6": {"instruction": "put the cream cheese in the bowl", "task_index": 13, "episodes": [384, 391, 393, 413, 434, 439, 457, 464, 491, 492, 508, 510, 518, 527, 536, 540, 562, 564, 581, 587, 600, 610, 632, 644, 650, 657, 677, 698, 709, 714, 720, 730, 748, 756, 760, 771, 776, 777, 787, 797]}, "7": {"instruction": "turn on the stove", "task_index": 16, "episodes": [388, 390, 406, 412, 418, 435, 440, 441, 450, 453, 460, 477, 486, 487, 499, 504, 520, 529, 531, 567, 569, 572, 575, 578, 602, 605, 609, 613, 617, 622, 628, 640, 654, 658, 690, 700, 705, 717, 723, 728, 729, 736, 746, 753, 759, 774, 783, 785, 799, 800]}, "8": {"instruction": "put the bowl on the plate", "task_index": 10, "episodes": [379, 422, 426, 431, 433, 447, 448, 451, 459, 466, 481, 483, 488, 507, 511, 513, 522, 532, 537, 549, 551, 563, 568, 582, 607, 615, 620, 621, 626, 634, 639, 642, 646, 653, 655, 670, 679, 708, 716, 718, 726, 727, 749, 750, 768, 770, 801, 803, 806]}, "9": {"instruction": "put the wine bottle on the rack", "task_index": 11, "episodes": [380, 381, 403, 407, 408, 428, 429, 467, 484, 485, 502, 503, 509, 517, 539, 542, 571, 579, 583, 586, 595, 623, 625, 651, 652, 666, 667, 671, 675, 683, 695, 719, 739, 757, 773, 798]}}}
EOF
cat > /tmp/select_episodes.py <<'EOF'
import json
from libero.libero import benchmark

p = {k: v.strip('"') for k, v in (l.strip().split("=", 1) for l in open("/tmp/params.sh") if "=" in l)}
m = json.load(open("/tmp/libero_goal_episodes.json"))
entry = m["tasks"][p["TASK"]]
instr = benchmark.get_benchmark_dict()[p["SUITE"]]().get_task(int(p["TASK"])).language
assert instr.strip().lower() == entry["instruction"], (instr, entry["instruction"])
print(f"instruction: {instr} | dataset task_index {entry['task_index']}")
print(f"episodes: {len(entry['episodes'])} | dataset revision: {m['revision']}")
json.dump(entry["episodes"], open("/tmp/r2_episodes.json", "w"))
open("/tmp/ds_rev", "w").write(m["revision"])
EOF
python /tmp/select_episodes.py 2>&1 | tail -3 | tee -a /kaggle/working/r2_report.txt
[ -s /tmp/r2_episodes.json ] && [ "$(cat /tmp/r2_episodes.json)" != "[]" ] || { echo "no episodes selected" | tee -a /kaggle/working/r2_report.txt; exit 1; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
mkdir -p /kaggle/working/r2/eval
# Evaluate one policy on one LIBERO task: eval_one GPU POLICY_PATH NAS TASK NAME [extra args]
cat > /tmp/eval_one.sh <<'EOF'
#!/usr/bin/env bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
GPU=$1; POL=$2; NAS=$3; T=$4; NAME=$5; shift 5
DIR=/kaggle/working/r2/eval/$NAME
[ -f $DIR/eval_info.json ] && exit 0
S=$(date +%s)
MUJOCO_EGL_DEVICE_ID=0 /tmp/venv/bin/python -m lerobot.scripts.lerobot_eval \
  --policy.path=$POL --policy.device=cuda:$GPU --policy.n_action_steps=$NAS "$@" \
  --env.type=libero --env.task=$SUITE "--env.task_ids=[$T]" --env.max_parallel_tasks=1 \
  --eval.n_episodes=$EVAL_EPISODES --eval.batch_size=5 --seed=1000 \
  --output_dir=$DIR > $DIR.log 2>&1
RC=$?
echo "[gpu$GPU] $NAME: exit=$RC seconds=$(( $(date +%s) - S )) $(grep -o '"pc_success": [0-9.]*' $DIR/eval_info.json 2>/dev/null | tail -1)" >> /kaggle/working/r2/progress.log
[ $RC -ne 0 ] && tail -15 $DIR.log >> /kaggle/working/r2/progress.log
exit 0
EOF
chmod +x /tmp/eval_one.sh
EPS=$(tr -d ' ' < /tmp/r2_episodes.json)

# GPU0: training (no env during training, so pin it to GPU0 only)
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i 0 -lms 2000 > /tmp/mem_train.txt &
MON=$!
S=$(date +%s)
CUDA_VISIBLE_DEVICES=0 /tmp/venv/bin/python -m lerobot.scripts.lerobot_train \
  --policy.path=lerobot/smolvla_base --policy.device=cuda --policy.push_to_hub=false \
  --dataset.repo_id=lerobot/libero --dataset.revision=$(cat /tmp/ds_rev) "--dataset.episodes=$EPS" \
  '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --batch_size=$BATCH --steps=$STEPS --save_freq=$SAVE_FREQ --log_freq=100 --seed=1000 \
  --output_dir=/kaggle/working/r2/train > /kaggle/working/r2/train.log 2>&1 &
TRAIN=$!

# GPU1 meanwhile: R1b (paper-architecture checkpoint, needs the camera rename), then R3
(
  if [ "$R1B" = "1" ]; then
    for T in 0 1 2 3 4 5 6 7 8 9; do
      /tmp/eval_one.sh 1 $R1B_CKPT $R1B_NAS $T r1b_task${T}_nas${R1B_NAS} '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}'
    done
  fi
  if [ "$R3" = "1" ]; then
    for T in $R3_TASKS; do for N in $R3_NAS; do
      /tmp/eval_one.sh 1 HuggingFaceVLA/smolvla_libero $N $T r3_task${T}_nas${N}
    done; done
  fi
) &
R3PID=$!

wait $TRAIN; RC=$?
kill $MON
echo "=== TRAIN exit=$RC minutes=$(( ($(date +%s) - S) / 60 )) peakMiB=$(sort -n /tmp/mem_train.txt | tail -1)" | tee -a /kaggle/working/r2_report.txt
grep -E "step:" /kaggle/working/r2/train.log | tail -3 | tee -a /kaggle/working/r2_report.txt
[ $RC -ne 0 ] && tail -30 /kaggle/working/r2/train.log | tee -a /kaggle/working/r2_report.txt
[ -n "$R3PID" ] && wait $R3PID
echo "R3 done" | tee -a /kaggle/working/r2_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
# Checkpoint sweep on task 0, split over both GPUs; the last checkpoint also with n_action_steps = 1
CKS=$(ls -d /kaggle/working/r2/train/checkpoints/*/pretrained_model 2>/dev/null | sort)
[ -n "$CKS" ] || { echo "no checkpoints" | tee -a /kaggle/working/r2_report.txt; exit 1; }
LAST=$(echo "$CKS" | tail -1)
i=0
for CK in $CKS; do
  STEP=$(basename $(dirname $CK))
  /tmp/eval_one.sh $((i % 2)) $CK $EVAL_NAS $TASK ckpt${STEP}_nas${EVAL_NAS} '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' &
  i=$((i + 1))
  [ $((i % 2)) -eq 0 ] && wait
done
wait
if [ "$FINAL_NAS1" = "1" ]; then
  /tmp/eval_one.sh 0 $LAST 1 $TASK ckpt$(basename $(dirname $LAST))_nas1 '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}'
fi
cat /kaggle/working/r2/progress.log | tee -a /kaggle/working/r2_report.txt
# Keep weights, drop optimizer state of all but the last checkpoint (output size)
for D in $(ls -d /kaggle/working/r2/train/checkpoints/*/training_state 2>/dev/null | sort | head -n -1); do rm -rf $D; done
du -sh /kaggle/working/r2/train | tee -a /kaggle/working/r2_report.txt

In [ ]:
import json, math, re
from pathlib import Path

out = Path("/kaggle/working/r2")

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * (c - h), 100 * (c + h))

def row(name):
    f = out / "eval" / name / "eval_info.json"
    if not f.exists():
        return f"| {name} | failed / missing | | | |"
    o = json.load(open(f))["overall"]
    k, n = int(o["n_success"]), int(o["n_episodes"])
    lo, hi = wilson(k, n)
    return f"| {name} | {k}/{n} | {100 * k / n:.0f} | {lo:.0f}–{hi:.0f} | {o.get('eval_ep_s', float('nan')):.0f} |"

log = (out / "train.log").read_text(errors="ignore") if (out / "train.log").exists() else ""
# LeRobot log order: step … loss … data_s … updt_s … step_s
steps = re.findall(r"step:(\S+).*?loss:(\S+).*?data_s:(\S+).*?step_s:(\S+)", log)
lines = ["# R2 · smolvla_base fine-tuned on LIBERO-Goal task 0", ""]
if steps:
    dat = [float(s[2]) for s in steps]; stp = [float(s[3]) for s in steps]
    lines += [f"- train log points: {len(steps)}; last: step {steps[-1][0]}, loss {steps[-1][1]}",
              f"- mean step time {sum(stp) / len(stp):.3f} s/step (data loading {sum(dat) / len(dat):.3f} s)", ""]
lines += ["| run | success | SR % | 95% Wilson CI | s/episode |", "|---|---|---|---|---|"]
names = sorted(p.name for p in (out / "eval").iterdir() if p.is_dir())
lines += [row(n) for n in names if n.startswith("ckpt")]
lines += ["", "**R3 · official smolvla_libero, Goal (n_action_steps = 1 is in R1)**", "",
          "| run | success | SR % | 95% Wilson CI | s/episode |", "|---|---|---|---|---|"]
lines += [row(n) for n in names if n.startswith("r3_")]
r1b = [n for n in names if n.startswith("r1b_")]
if r1b:
    K = N = 0
    for n in r1b:
        f = out / "eval" / n / "eval_info.json"
        if f.exists():
            o = json.load(open(f))["overall"]; K += int(o["n_success"]); N += int(o["n_episodes"])
    lo, hi = wilson(K, N)
    lines += ["", "**R1b · lerobot/smolvla_libero (paper architecture), Goal**", "",
              "| run | success | SR % | 95% Wilson CI | s/episode |", "|---|---|---|---|---|"]
    lines += [row(n) for n in sorted(r1b, key=lambda s: int(s.split("_")[1][4:]))]
    lines += [f"| **suite** | **{K}/{N}** | **{100 * K / max(N, 1):.1f}** | **{lo:.1f}–{hi:.1f}** | |"]
(out / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))